### 1. The Artificial Neuron (The Building Block)

Deep Learning ka base ek biological neuron se inspire hokar banaya gaya mathematical abstraction hai. Ek artificial neuron fundamentally ek function hai jo 3 steps me kaam karta hai:

#### A. Inputs ($x$) aur Weights ($w$)
Maan lo hum predict kar rahe hain ki ek student exam me pass hoga ya nahi. 
Inputs: $x_1$ = Hours studied, $x_2$ = Hours slept.
Har input ka impact alag hota hai. Is "importance" ko hum **Weights ($w$)** bolte hain. Network ko yehi weights seekhne hote hain.

#### B. The Linear Summation (Dot Product)
Neuron har input ko uske weight se multiply karta hai aur sabko add kar deta hai. Saath me ek extra constant **Bias ($b$)** lagata hai (jo decision boundary ko shift karne me madad karta hai).

$$z = (w_1x_1 + w_2x_2 + \dots + w_nx_n) + b$$

Vectors ki form me parallel processing ke liye ye dot product ban jata hai:

$$z = \mathbf{w}^T \mathbf{x} + b$$

#### C. The Activation Function ($f$)
$z$ ek pure *linear calculation* hai. Real world problems *non-linear* hoti hain. Isliye hum $z$ ko ek Activation Function (jaise ReLU, Sigmoid) se guzarte hain taaki model tedi-medi lines aur complex patterns seekh sake.

$$\hat{y} = f(z)$$

---

### 2. What is "Deep Learning"? (The MIT Definition)

MIT 6.S191 Deep Learning ko do alag nazariyon se define karta hai, aur real magic dono ke combination me hai:

#### Definition 1: The Architecture Perspective
> "Neural nets: A class of machine learning architectures that use stacks of linear transformations interleaved with pointwise nonlinearities."

* **"Linear transformations":** Multiple neurons ke calculations Matrix Algebra ban jate hain: $\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$.
* **"Interleaved with pointwise nonlinearities":** Matrices ke calculations par element-by-element non-linear functions (jaise ReLU) lagana, theek ek sandwich ki tarah.
* **"Stacks":** In layers ko ek ke upar ek rakhna jisse architecture "Deep" banta hai. Agar hum non-linearities hata dein, toh pure network ki math collapse hokar wapas single matrix multiplication ban jayegi.

#### Definition 2: The Optimization Perspective (Differentiable Programming)
> "Differentiable programming: A programming paradigm where parameterize parts of the program and let gradient-based optimization tune the parameters."

Humne architecture toh bana liya, par model "seekhega" kaise? Yahan aati hai deep math ki Calculus:

* **"Parameterized parts of the program":** Hamare network me $\mathbf{W}$ (weights) aur $\mathbf{b}$ (biases) wo parameters hain jinhe hum tune kar sakte hain. Shuru me inki value random hoti hai.
* **The Loss Function ($L$):** Model ne kitni galti ki, ye measure karne ke liye hum ek function banate hain. Jaise Mean Squared Error:
  $$L = \frac{1}{n} \sum (y - \hat{y})^2$$
* **"Gradient-based optimization" (Backpropagation):** Hum Calculus ka **Chain Rule** use karke Loss ka derivative (Gradient) nikalte hain har ek weight ke respect me ($\frac{\partial L}{\partial w}$). Ye gradient batata hai ki weight ko kis direction me thoda sa change karein ki Loss kam ho jaye.
* **Tuning the Parameters (Gradient Descent):** Ab model apne weights update karta hai $\alpha$ (learning rate) ka use karke:
  $$w_{new} = w_{old} - \alpha \frac{\partial L}{\partial w}$$

---

### 3. PyTorch Implementation (Architecture + Optimization)

Neeche diya gaya code purely upar di gayi dono definitions ko follow karta hai.



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ==========================================
# DEFINITION 1: NEURAL NET ARCHITECTURE
# Stacks of linear transformations & nonlinearities
# ==========================================
class DeepMathNet(nn.Module):
    def __init__(self):
        super(DeepMathNet, self).__init__()
        # Parameterized parts (Weights W aur Biases b)
        self.layer1 = nn.Linear(in_features=2, out_features=4) 
        self.layer2 = nn.Linear(in_features=4, out_features=4)
        self.layer3 = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        # Stacking linear transformations (z) and pointwise nonlinearities (F.relu)
        z1 = self.layer1(x)
        h1 = F.relu(z1)      # Interleaved non-linearity
        
        z2 = self.layer2(h1)
        h2 = F.relu(z2)      # Interleaved non-linearity
        
        z3 = self.layer3(h2)
        output = torch.sigmoid(z3) # Output probability between 0 and 1
        
        return output

# ==========================================
# GPU / CPU DEVICE SELECTION
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# Dummy Input (x) and Target Label (y)
x_input = torch.tensor([[5.0, 7.0]]).to(device) # [Hours Studied, Hours Slept]
y_true = torch.tensor([[1.0]]).to(device)       # Target: 1 (Pass)

# ==========================================
# DEFINITION 2: DIFFERENTIABLE PROGRAMMING
# Gradient-based optimization to tune parameters
# ==========================================

model = DeepMathNet().to(device)

# Loss Function: Binary Cross Entropy (Model kitna galat hai)
criterion = nn.BCELoss()

# Optimizer: Stochastic Gradient Descent (Gradient based optimization rule)
# Learning rate (alpha) = 0.01
optimizer = optim.SGD(model.parameters(), lr=0.01)

# --- A SINGLE TRAINING STEP ---

# 1. Forward Pass (Make a prediction)
y_pred = model(x_input)

# 2. Calculate Loss (Measure the error)
loss = criterion(y_pred, y_true)

# 3. Calculate Gradients (Calculus chain rule inside PyTorch)
# Calculate dL/dW for all weights
loss.backward()

# 4. Tune the parameters! (Apply W_new = W_old - alpha * dL/dW)
optimizer.step()

# Reset gradients for the next loop
optimizer.zero_grad()

print(f"Prediction before fully training: {y_pred.item():.4f}")
print(f"Current Loss: {loss.item():.4f}")
print("Gradients have been used to update parameters successfully!")

Using Device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
Prediction before fully training: 0.6157
Current Loss: 0.4849
Gradients have been used to update parameters successfully!
